# Kangaroo secp256k1 multi-GPU (Colab + Kaggle)
**Colab** : Exécution → Modifier le type d'exécution → *T4 GPU*.
**Kaggle** : Settings → Accelerator → *GPU T4 x2* et Internet *On*.

In [ ]:
!nvidia-smi -L

## 1. Dossier de travail
Colab : Google Drive. Kaggle : `/kaggle/working`. Les fichiers `tame_NNN.dp` y sont écrits.

In [ ]:
import os
if os.path.exists('/kaggle'):
    WORK = '/kaggle/working'
else:
    from google.colab import drive
    drive.mount('/content/drive')
    WORK = '/content/drive/MyDrive/kangaroo'
os.makedirs(WORK, exist_ok=True)
%cd $WORK

## 2. Récupération du code (GitHub) + compilation

In [ ]:
%%bash
rm -rf /tmp/src && git clone -q https://github.com/lawahakassou-tech/cuda.git /tmp/src
cd /tmp/src && wc -l kangaroo.cu
cc=$(nvidia-smi --query-gpu=compute_cap --format=csv,noheader | head -1 | tr -d .)
NVCC=$(which nvcc || echo /usr/local/cuda/bin/nvcc)
echo "architecture sm_$cc"
$NVCC -O3 -std=c++17 -arch=sm_$cc -Xcompiler -pthread kangaroo.cu -o /tmp/kangaroo 2>&1 | grep -iE "error" || echo "compilation OK"

## 3. Autotests (tous les GPU visibles) — doit finir par « Test GPU : tout est OK »

In [ ]:
!/tmp/kangaroo --selftest | tail -1
!/tmp/kangaroo --gpucheck

## 4. Mode 1 : générer les points Tame
`--gpu` = tous les GPU. `--gpus 1` pour n'en utiliser qu'un. `-n 0` = jusqu'à l'arrêt de la cellule.

In [ ]:
!/tmp/kangaroo -m 1 -r 80 -d 20 -n 0 --gpu -y --save 120

## 5. Mode 2 : continuer un fichier Tame

In [ ]:
!/tmp/kangaroo -m 2 -f 1 -n 0 --gpu -y --save 120

## 6. Mode 3 : résoudre une clé publique

In [ ]:
PUB = "02..."   # <- clé publique (hex)
!/tmp/kangaroo -m 3 -f 1 -k {PUB} --gpu -y